## **Goal**
**Given a image of a human pointing, use BEV to generate:**
* SMPL parameters
* 3D joint positions in camera coordinates (Unreal Engine format)

### **Setup**
```
conda create -n fbv-bev python=3.9
conda activate fbv-bev
```
https://github.com/Arthur151/ROMP/blob/master/simple_romp/README.md
```
conda install matplotlib
conda install ipykernel=6.29.5
pip install PyQt5
```

In [16]:
import bev
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [17]:
%matplotlib qt

### **BEV Initialization**

In [18]:
# Configure BEV Settings

settings = bev.main.default_settings
settings.mode = 'image'
settings.render_mesh = False  # Skip the 3D mesh creation
settings.show = False         # Skip opening a display window
settings.save_dict = False    # Skip saving .npz files to disk

In [19]:
# Load BEV Model

bev_model = bev.BEV(settings)

Using BEV.
Threshold for positive center detection: 0.08


In [20]:
def get_3d_joints(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    outputs = bev_model(img)
    
    if outputs is not None and 'joints' in outputs:
        data = outputs['joints']
        if hasattr(data, 'detach'):
            return data.detach().cpu().numpy()
        return data
        
    return None

### **Run Inference**

In [21]:
# Run Model On Given File

image_file = "00002A.png"
joints = get_3d_joints(image_file)

if joints is not None:
    print(f"Detected {len(joints)} people.")
    print(f"First person Pelvis (X, Y, Z): {joints[0][0]}")

Detected 1 people.
First person Pelvis (X, Y, Z): [-0.00847259  0.01942791  0.00653743]


### **Visualization**

In [22]:
def plot_bev_skeleton(joints_3d, person_idx=0):
    # 1. Sanitize data
    person_joints = np.array(joints_3d[person_idx]).astype(float)
    if np.isnan(person_joints).any():
        person_joints = np.nan_to_num(person_joints)

    # 2. Connections
    smpl_connections = [
        (0, 1), (0, 2), (0, 3), (1, 4), (4, 7), (7, 10), (2, 5), (5, 8), (8, 11),
        (3, 6), (6, 9), (9, 12), (12, 15), (9, 13), (13, 16), (16, 18), (18, 20),
        (20, 22), (9, 14), (14, 17), (17, 19), (19, 21), (21, 23)
    ]

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Data mapping (Camera to Plotting space)
    x = person_joints[:, 0]
    y = -person_joints[:, 1] # Flip Y so head is up
    z = person_joints[:, 2]

    # 3. Calculate Midpoints and Range for "Equal" scaling
    mid_x = (x.max() + x.min()) * 0.5
    mid_y = (y.max() + y.min()) * 0.5
    mid_z = (z.max() + z.min()) * 0.5
    max_range = np.array([x.max()-x.min(), y.max()-y.min(), z.max()-z.min()]).max() / 2.0

    # Apply limits to all axes equally
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_z - max_range, mid_z + max_range) # Z mapped to Y-axis
    ax.set_zlim(mid_y - max_range, mid_y + max_range) # -Y mapped to Z-axis

    # Force the visual box to be a cube
    ax.set_box_aspect([1,1,1]) 

    # 4. Plot
    ax.scatter(x, z, y, c='blue', s=20)
    for start, end in smpl_connections:
        ax.plot([x[start], x[end]], [z[start], z[end]], [y[start], y[end]], c='red', linewidth=2)

    ax.set_xlabel('X (Lateral)')
    ax.set_ylabel('Z (Depth)')
    ax.set_zlabel('Y (Height)')
    
    plt.show()

In [23]:
if joints is not None:
    plot_bev_skeleton(joints, person_idx=0)